# فصل 16: پروژه‌ی نهایی یکپارچه

کد این نوت‌بوک عیناً از فصل 16 کتاب «مهندسی هوش مصنوعی مکانی» برداشته شده است.

پیش از اجرا: مسیر داده‌ها را با ساختار پوشه‌ی `data/`، `work/` و `out/` هماهنگ کنید. راه دانلود هر فایل در پیوست ب کتاب آمده است.


## ۱۶-۲) لایه‌ی داده: خواندن هوشمند تصویر


In [ ]:
import rasterio
from rasterio.windows import Window

with rasterio.open("data/B04.tif") as src:
    print(src.crs, src.res)     # e.g. EPSG:32639, (10.0, 10.0)

    # read only a small 256x256 window, not the whole image
    win = Window(col_off=100, row_off=100, width=256, height=256)
    patch = src.read(1, window=win)

## ۱۶-۴) لایه‌ی مدل: آموزش و ذخیره‌ی صادقانه


In [ ]:
import joblib, json
from datetime import datetime, timezone
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)
model.fit(X_train, y_train)

version = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
joblib.dump(model, f"models/rf_flood_{version}.joblib")

meta = {
    "version": version,
    "features": feature_names,
    "accuracy": float(acc),
    "trained_at": version,
}
json.dump(meta, open(f"models/rf_flood_{version}.json", "w"), indent=2)

## ۱۶-۵) لایه‌ی API: سرو کردن مدل


In [ ]:
from fastapi import APIRouter, HTTPException, Query
import joblib

router = APIRouter(prefix="/flood", tags=["flood"])
model = joblib.load("models/rf_flood_v1.joblib")   # load ONCE at startup
LEVELS = {0: "low", 1: "medium", 2: "high"}

@router.get("/classify-point")                     # a read-only query -> GET
def classify_point(lat: float = Query(..., ge=-90, le=90),
                   lon: float = Query(..., ge=-180, le=180)):
    try:
        features = extract_features_at_point(lat, lon)   # window read
    except OutOfImageError:
        raise HTTPException(400, "point is outside the image")
    probs = model.predict_proba([features])[0]
    cls = int(probs.argmax())
    return {
        "lat": lat,
        "lon": lon,
        "risk_level": LEVELS[cls],
        "probabilities": {LEVELS[i]: round(float(pr), 3)
                          for i, pr in enumerate(probs)},
    }